In [4]:
import pandas as pd
import re
import json

In [5]:
def load_reviews(path):
    if path.endswith(".jsonl"):
        rows = [json.loads(line) for line in open(path)]
        return pd.DataFrame(rows)
    return pd.read_parquet(path)
 
reviews = load_reviews("reviews.parquet")  # your real Glassdoor reviews file
 
print(f"Loaded {len(reviews)} reviews")
print(reviews.columns.tolist())

Loaded 1543 reviews
['company_name', 'department', 'occupation', 'page', 'rating', 'date', 'title', 'job_title', 'employment_status', 'location', 'recommend', 'ceo_approval', 'business_outlook', 'pros', 'cons', 'city', 'state', 'region']


In [6]:
def clean_text(t):
    if pd.isna(t) or t is None:
        return ""
    t = str(t).lower()
    t = re.sub(r"[^a-z0-9\s'/-]", " ", t)   # strip weird punctuation, keep words
    t = re.sub(r"\s+", " ", t).strip()
    return t
 
reviews["pros_clean"] = reviews["pros"].apply(clean_text)
reviews["cons_clean"] = reviews["cons"].apply(clean_text)
reviews["rating"] = pd.to_numeric(reviews["rating"], errors="coerce")

In [7]:
LEXICONS = {
    "burnout_exhaustion": [
        "exhausted", "exhausting", "burnout", "burnt out", "burned out", "no break",
        "understaffed", "short staffed", "long hours", "24/7", "never off the clock",
        "overworked", "no lunch", "unsustainable", "extended school day",
        "extended school year", "excessive work", "zero work-life balance",
        "not enough employees", "sitting a lot", "on call", "getting called off",
    ],
    "compassion_fatigue": [
        "difficult students", "troubled", "behavioral issues", "trauma",
        "hard to leave it at work", "emotionally draining", "tough behaviors",
        "high needs", "disrespectful students", "lots of patients", "patient ratio",
        "acuity", "12:1",
    ],
    "management_frustration": [
        "no support", "lack of support", "poor communication", "communication is lacking",
        "micromanage", "unprofessional", "out of touch", "arbitrary decisions",
        "leadership", "administration", "no real salary schedule", "money hungry",
        "don't listen", "don't follow it", "unwelcoming",
    ],
    "fulfillment_purpose": [
        "rewarding", "love the kids", "love my job", "make a difference", "inspiring",
        "amazing", "fun to work with", "supportive team", "great environment",
        "genuinely love their job", "empowered",
    ],
    "salary_dissatisfaction": [
        "underpaid", "low pay", "paycheck", "no real salary", "hourly pay",
        "paid about", "slashed", "no pay in the summer", "pay is",
        "pay could be more competitive", "high cost for insurance",
    ],
    "turnover_instability": [
        "turnover", "revolving door", "staff turnover", "high turnover",
        "advance is difficult", "stay in their positions for a long time",
    ],
    # --- Positive constructs (balance out the negative ones above) ---
    "job_satisfaction": [
        "great place to work", "good place to work", "great job", "highly recommend",
        "would recommend", "enjoy my job", "happy here", "positive experience",
        "good company to work for",
    ],
    "team_support": [
        "supportive colleagues", "great colleagues", "team player", "we are all one big family",
        "everyone supports one another", "hardworking colleagues", "close-knit",
        "great co-workers", "helpful staff", "collaboration", "mentorship",
    ],
    "pride_recognition": [
        "proud", "appreciated", "valued", "recognized", "impact", "student growth",
        "grow professionally",
    ],
    "healthy_balance": [
        "flexible", "work life balance", "work-life balance", "summers off",
        "reasonable hours", "short days", "good hours", "time off",
        "customizable schedule", "decent pay",
    ],
}
 

In [8]:
def score_construct(text, keywords):
    """Returns (hit_count, hit_flag, matched_terms)."""
    hits = [kw for kw in keywords if kw in text]
    return len(hits), int(len(hits) > 0), hits

In [9]:
reviews["full_text"] = reviews["pros_clean"] + " " + reviews["cons_clean"]
 
for construct, keywords in LEXICONS.items():
    counts, flags, terms = [], [], []
    for text in reviews["full_text"]:
        c, f, t = score_construct(text, keywords)
        counts.append(c)
        flags.append(f)
        terms.append(", ".join(t))
    reviews[f"{construct}_count"] = counts
    reviews[f"{construct}_flag"] = flags
    reviews[f"{construct}_terms"] = terms

In [10]:
POSITIVE_WORDS = {"great", "good", "amazing", "supportive", "rewarding", "love",
                   "excellent", "friendly", "helpful", "flexible", "fun", "inspiring"}
NEGATIVE_WORDS = {"bad", "poor", "unprofessional", "toxic", "hostile", "awful",
                   "difficult", "demanding", "unsustainable", "disrespectful", "lacking"}
 
def polarity(text):
    words = text.split()
    if not words:
        return 0.0
    pos = sum(1 for w in words if w in POSITIVE_WORDS)
    neg = sum(1 for w in words if w in NEGATIVE_WORDS)
    return (pos - neg) / len(words)
 
reviews["pros_sentiment"] = reviews["pros_clean"].apply(polarity)
reviews["cons_sentiment"] = reviews["cons_clean"].apply(polarity)

In [13]:
reviews["invisible_labor_score"] = (
    reviews["burnout_exhaustion_count"] * 1.0
    + reviews["compassion_fatigue_count"] * 0.75
    + reviews["management_frustration_count"] * 0.5
    - reviews["fulfillment_purpose_count"] * 0.5
)
 
# Positive-side composite - job satisfaction, support, pride, healthy balance.
# Keep this separate rather than just inverting the negative score - a review
# can score high on BOTH (burnt out but still proud of the work), and that
# combination is itself an interesting finding for your report.
reviews["positive_experience_score"] = (
    reviews["job_satisfaction_count"] * 1.0
    + reviews["team_support_count"] * 0.75
    + reviews["pride_recognition_count"] * 0.75
    + reviews["healthy_balance_count"] * 0.5
    + reviews["fulfillment_purpose_count"] * 0.5
)

In [14]:
POSITIVE_WORDS = {"great", "good", "amazing", "supportive", "rewarding", "love",
                   "excellent", "friendly", "helpful", "flexible", "fun", "inspiring"}
NEGATIVE_WORDS = {"bad", "poor", "unprofessional", "toxic", "hostile", "awful",
                   "difficult", "demanding", "unsustainable", "disrespectful", "lacking"}
 
def polarity(text):
    words = text.split()
    if not words:
        return 0.0
    pos = sum(1 for w in words if w in POSITIVE_WORDS)
    neg = sum(1 for w in words if w in NEGATIVE_WORDS)
    return (pos - neg) / len(words)
 
reviews["pros_sentiment"] = reviews["pros_clean"].apply(polarity)
reviews["cons_sentiment"] = reviews["cons_clean"].apply(polarity)

In [15]:
# ---------------------------------------------------------------------------
# STEP 7: SAVE PER-REVIEW OUTPUT
# ---------------------------------------------------------------------------
review_cols = ["company_name", "occupation", "region", "rating", "recommend",
               "burnout_exhaustion_flag", "compassion_fatigue_flag",
               "management_frustration_flag", "fulfillment_purpose_flag",
               "salary_dissatisfaction_flag", "turnover_instability_flag",
               "job_satisfaction_flag", "team_support_flag",
               "pride_recognition_flag", "healthy_balance_flag",
               "pros_sentiment", "cons_sentiment",
               "invisible_labor_score", "positive_experience_score"]
reviews[review_cols].to_csv("classified_reviews.csv", index=False)
print("\nSaved classified_reviews.csv")
 


Saved classified_reviews.csv


In [16]:
summary = reviews.groupby(["occupation", "region"]).agg(
    n_reviews=("rating", "count"),
    avg_rating=("rating", "mean"),
    burnout_rate=("burnout_exhaustion_flag", "mean"),
    compassion_fatigue_rate=("compassion_fatigue_flag", "mean"),
    fulfillment_rate=("fulfillment_purpose_flag", "mean"),
    job_satisfaction_rate=("job_satisfaction_flag", "mean"),
    team_support_rate=("team_support_flag", "mean"),
    healthy_balance_rate=("healthy_balance_flag", "mean"),
    avg_invisible_labor_score=("invisible_labor_score", "mean"),
    avg_positive_experience_score=("positive_experience_score", "mean"),
).reset_index()
 
summary.to_csv("occupation_region_summary.csv", index=False)
print("Saved occupation_region_summary.csv")
 
print("\n--- PREVIEW ---")
print(reviews[review_cols].to_string())
print("\n--- SUMMARY ---")
print(summary.to_string())

Saved occupation_region_summary.csv

--- PREVIEW ---
                                        company_name              occupation     region  rating recommend  burnout_exhaustion_flag  compassion_fatigue_flag  management_frustration_flag  fulfillment_purpose_flag  salary_dissatisfaction_flag  turnover_instability_flag  job_satisfaction_flag  team_support_flag  pride_recognition_flag  healthy_balance_flag  pros_sentiment  cons_sentiment  invisible_labor_score  positive_experience_score
0                    HealthTrust Workforce Solutions                   Nurse       None     4.0  positive                        0                        0                            0                         0                            0                          0                      0                  0                       0                     0        0.066667       -0.090909                   0.00                       0.00
1                    HealthTrust Workforce Solutions                   Nu